In [1]:
!pip install -q chromadb
!pip install -U -q "google-genai"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-exporter-otlp-proto-http 1.37.0 requires opentelemetry-exporter-otlp-proto-common==1.37.0, but you have opentelemetry-exporter-otlp-proto-common 1.38.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.37.0 requires opentelemetry-proto==1.37.0, but you have opentelemetry-proto 1.38.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.37.0 requires opentelemetry-sdk~=1.37.0, but you have opentelemetry-sdk 1.38.0 which is incompatible.
google-adk 1.19.0 requires opentelemetry-api<=1.37.0,>=1.37.0, but you have opentelemetry-api 1.38.0 which is incompatible.
google-adk 1.19.0 requires opentelemetry-sdk<=1.37.0,>=1.37.0, but you have opentelemetry-sdk 1.38.0 which is incompatible.


In [2]:
from google import genai
import os
from dotenv import load_dotenv

load_dotenv()

# El cliente de Gemini para hacer los embedding
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
client = genai.Client(api_key=GEMINI_API_KEY)

## API de TMDB para conseguir información de películas y cast

In [3]:
import json
import requests

# Para obtener información de una película
# usamos la API de TMDB
TMDB_API_KEY = os.getenv('TMDB_API_KEY')
TMDB_HEADERS = {
      "accept": "application/json",
      "Authorization": f"Bearer {TMDB_API_KEY}"
}

def get_movies_info(title):
  url = f"https://api.themoviedb.org/3/search/movie?query={title}&include_adult=false&language=en-US&page=1"

  response = requests.get(url, headers=TMDB_HEADERS)
  return json.loads(response.text)

def get_movie_reviews(movie_id):
  url = f"https://api.themoviedb.org/3/movie/{movie_id}/reviews?language=en-US&page=1"

  response = requests.get(url, headers=TMDB_HEADERS)
  return json.loads(response.text)

def get_movie_cast(movie_id):
  url = f"https://api.themoviedb.org/3/movie/{movie_id}/credits?language=en-US"

  response = requests.get(url, headers=TMDB_HEADERS)
  return json.loads(response.text)

def get_person_details(person_id):
  url = f"https://api.themoviedb.org/3/person/{person_id}"

  response = requests.get(url, headers=TMDB_HEADERS)
  return json.loads(response.text)


## Añadir películas a la colección "movies" de ChromaDB

In [4]:
import chromadb
chroma_client = chromadb.PersistentClient(path="chroma_db")

# Función para añadir películas encontradas a la colección
# la colección es una parte de la base de datos
# hay colecciones por categorías (películas, actores, reviews...)
def add_movies_to_collection(movies_info):
  movies_col = chroma_client.get_or_create_collection('movies')

  ids = []
  embeddings = []
  metadatas = []
  documents = []

  def get_cast_info(movie):
    cast_found = get_movie_cast(str(movie['id']))
    director = "";
    cast = "";

    for person in cast_found['cast']:
      if person['known_for_department'] == 'Directing':
        director += person['name'] + ", "
      else:
        cast += person['name'] + ", "
    return director, cast

  def get_metadata(movie, director, cast):
    return {
        "movie_title" : movie['title'],
        "director"    : director,
        "cast"        : cast,
        "popularity"  : movie['popularity'],
        "release_date": movie['release_date'],
        "vote_average": movie['vote_average'],
        "vote_count"  : movie['vote_count']
    }

  for movie in movies_info['results']:
    # Consultamos con nuestra colección
    result = movies_col.get(
      ids=[str(movie['id'])],
      include=[]
    )

    # Si no existe, la añade
    if not result['ids'] and movie['overview']:
      # Primero obtenemos al cast
      director, cast = get_cast_info(movie)

      content_embeddings = client.models.embed_content(model="gemini-embedding-001",
                                                        contents=movie['overview']).embeddings[0]
      ids.append(str(movie['id']))
      embeddings.append(content_embeddings.values)
      metadatas.append(get_metadata(movie, director, cast))
      documents.append(movie['overview'])

  if len(ids) > 0:
    movies_col.add(
        ids = ids,
        embeddings = embeddings,
        metadatas = metadatas,
        documents = documents
    )

  print(f"Added {len(ids)} movies.")

# Ejemplo de cómo usarlo junto a la búsqueda en TMDB
movies = get_movies_info('mario the movie')
add_movies_to_collection(movies)

Added 0 movies.


## Añadir reviews a la colección "reviews" de ChromaDB

In [5]:
def add_reviews_to_collection(movie_id):
  reviews_col = chroma_client.get_or_create_collection('reviews')

  ids = []
  embeddings = []
  metadatas = []
  documents = []

  reviews = get_movie_reviews(movie_id)

  def get_metadata(review):
    if review["author_details"]["rating"]:
      return {
          "author"  : review["author"],
          "rating"  : review["author_details"]["rating"]
      }
    else:
      return {
          "author"  : review["author"],
      }

  for review in reviews['results']:

    # Consultamos con nuestra colección
    result = reviews_col.get(
      ids=[str(review['id'])],
      include=[]
    )
    # Si no existe, la añade
    if not result['ids'] and review['content']:
      ids.append(review['id'])
      metadatas.append(get_metadata(review))

      content_embeddings = client.models.embed_content(model="gemini-embedding-001",
                                                        contents=review['content']).embeddings[0]
      embeddings.append(content_embeddings.values)
      documents.append(review['content'])

      if len(ids) > 0:
        reviews_col.add(
          ids = ids,
          embeddings = embeddings,
          metadatas = metadatas,
          documents = documents
        )

  print(f"Added {len(ids)} reviews.")

# Ejemplo con la película 99 de TMDB (Todo Sobre Mi Madre)
add_reviews_to_collection("99")


Added 0 reviews.


## Añadir actores a la colección "people" de ChromaDB

In [6]:
def add_person_to_collection(person_id):
  people_col = chroma_client.get_or_create_collection('people')

  ids = []
  embeddings = []
  metadatas = []
  documents = []

  details = get_person_details(person_id)

  # Consultamos con nuestra colección
  result = people_col.get(
    ids=[str(details['id'])],
    include=[]
  )

  if not result['ids'] and details['biography']:
    ids.append(str(details['id']))

    gender = "Not set"
    if details['gender'] == 1:
      gender = "female"
    elif details['gender'] == 2:
      gender = "male"
    elif details['gender'] == 3:
      gender = "non binary"

    metadatas.append({'name': details['name'], 'department': details['known_for_department'], 'gender': gender})
    content_embeddings = client.models.embed_content(model="gemini-embedding-001",
                                                      contents=details['biography']).embeddings[0]
    embeddings.append(content_embeddings.values)
    documents.append(details['biography'])

    if len(ids) > 0:
      people_col.add(
      ids = ids,
      embeddings = embeddings,
      metadatas = metadatas,
      documents = documents
    )

    print(f"Added {details['name']}")
  else:
    print("Person already exists in collection")

# Ejemplo con persona 31 (Tom Hanks)
add_person_to_collection("31")


Person already exists in collection


## Ejemplo de consulta a la colección "movies" de ChromaDB

In [7]:
# Ejemplo de hacerle una pregunta a la colección
query = "película sobre coches"

# Hay que hacer un embedding porque hemos no usamos el modelo
# nativo de chromadb, sino el de gemini al meterlos en la colección
query_embedding = client.models.embed_content(model="gemini-embedding-001",
                                              contents=query).embeddings[0]
movies_col = chroma_client.get_or_create_collection('movies')

res = movies_col.query(
    query_embeddings=[query_embedding.values],
    n_results=3
)

res['metadatas'][0]

[{'release_date': '1954-11-01',
  'cast': "John Ireland, Dorothy Malone, Bruce Carlisle, Iris Adrian, Marshall Bradford, Bruno VeSota, Byrd Holland, Larry Thor, Henry Rowland, Dick Pinner, Robin Morse, Harry 'Snub' Pollard, Jean Howell, Roger Corman, Jonathan Haze, William Woodson, ",
  'popularity': 5.6531,
  'vote_average': 4.9,
  'vote_count': 33,
  'director': 'Lou Place, ',
  'movie_title': 'The Fast and the Furious'},
 {'director': '',
  'vote_count': 74,
  'cast': 'Steve Clemmons, Charlyne Yi, ',
  'popularity': 1.0202,
  'vote_average': 7.9,
  'movie_title': 'Fast',
  'release_date': '2010-06-24'},
 {'movie_title': 'The Fast and the Furious',
  'popularity': 0.9461,
  'vote_average': 6.997,
  'release_date': '2001-06-22',
  'cast': 'Paul Walker, Vin Diesel, Michelle Rodriguez, Jordana Brewster, Rick Yune, Chad Lindberg, Johnny Strong, Matt Schulze, Ja Rule, Ted Levine, Thom Barry, Vyto Ruginis, Stanton Rutledge, Noel Gugliemi, R.J. de Vera, Beau Holden, Reggie Lee, David Dougla

---
<h2>Creacion de Agentes con Google Agent Development Kit</h2>


In [8]:
# Instalar librerías necesarias
!pip install google-adk litellm -q

import os
import requests
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

# API Key y configuración
#os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"
MODEL_GEMINI_2_0_FLASH = "gemini-2.0-flash"

print("Configuración ADK completada.")



ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-exporter-otlp-proto-grpc 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.37.0 which is incompatible.
opentelemetry-exporter-otlp-proto-grpc 1.38.0 requires opentelemetry-proto==1.38.0, but you have opentelemetry-proto 1.37.0 which is incompatible.
opentelemetry-exporter-otlp-proto-grpc 1.38.0 requires opentelemetry-sdk~=1.38.0, but you have opentelemetry-sdk 1.37.0 which is incompatible.
Configuración ADK completada.


In [9]:
#Tool que ya existe para buscar en WikiPedia
!pip install wikipedia -q
import wikipedia

import wikipedia

def buscar_en_wikipedia(titulo: str):
    try:
        query = f"{titulo} (film)"
        summary = wikipedia.summary(query, sentences=3)
        url = wikipedia.page(query).url
        return {"summary": summary, "url": url}
    except wikipedia.exceptions.DisambiguationError as e:
        # Si hay ambigüedad, elegir la opción que contenga "film" o "película"
        for option in e.options:
            if "film" in option.lower() or "película" in option.lower():
                summary = wikipedia.summary(option, sentences=3)
                url = wikipedia.page(option).url
                return {"summary": summary, "url": url}
        return f"No se encontró la página de la película para '{titulo}'."
    except Exception:
        return f"No se encontró información en Wikipedia para '{titulo}'."




In [10]:
def buscar_y_guardar_peliculas(titulo: str):
    pelis = get_movies_info(titulo)
    add_movies_to_collection(pelis)
    return f"Se agregaron {len(pelis.get('results', []))} películas de '{titulo}' a ChromaDB."

def obtener_peliculas_por_titulo(titulo: str):
    # Consultar la colección 'movies' de ChromaDB 
    movies_col = chroma_client.get_or_create_collection('movies')
    query_embedding = client.models.embed_content(model="gemini-embedding-001",
                                                  contents=titulo).embeddings[0].values
    res = movies_col.query(query_embeddings=[query_embedding], n_results=1)
    if res['metadatas'] and res['metadatas'][0]:
        return res['metadatas'][0][0]  # Devuelve la metadata de la primera película
    # Si no hay resultados, mirar en Wikipedia
    wiki = buscar_en_wikipedia(titulo)
    return wiki if isinstance(wiki, dict) else f"No se encontró información sobre '{titulo}'."

def obtener_detalles_pelicula(titulo: str):
    info = obtener_peliculas_por_titulo(titulo)
    if isinstance(info, dict) and 'movie_title' in info:
        return {
            "titulo": info.get("movie_title"),
            "director": info.get("director"),
            "cast": info.get("cast"),
            "fecha_estreno": info.get("release_date")
        }
    return info  # Devuelve Wikipedia o mensaje de error

def obtener_sinopsis_pelicula(titulo: str):
    info = obtener_peliculas_por_titulo(titulo)
    if isinstance(info, dict) and 'overview' in info:
        return info['overview']
    # Si no hay sinopsis, mirar Wikipedia
    wiki = buscar_en_wikipedia(titulo)
    if isinstance(wiki, dict):
        return wiki.get("summary", "Sinopsis no disponible.")
    return f"No se encontró información sobre '{titulo}'."

def obtener_detalles_actor(nombre: str):
    """
    Obtiene información detallada de un actor por su nombre.
    Primero intenta en TMDB y, si no encuentra resultados, busca en Wikipedia.
    """
    # 1 Buscar actor en TMDB
    url_search = f"https://api.themoviedb.org/3/search/person?query={nombre}&include_adult=false&language=en-US&page=1"
    response_search = requests.get(url_search, headers=TMDB_HEADERS)
    data_search = response_search.json()

    if data_search.get('results'):
        # Tomamos el primer resultado
        actor = data_search['results'][0]
        person_id = actor['id']

        # 2 Obtener detalles del actor usando la función existente
        detalles = get_person_details(person_id)
        gender = "Not set"
        if detalles['gender'] == 1:
            gender = "female"
        elif detalles['gender'] == 2:
            gender = "male"
        elif detalles['gender'] == 3:
            gender = "non binary"
        return {
            "nombre": detalles.get('name'),
            "biografia": detalles.get('biography') or "Sin biografía disponible",
            "fecha_nacimiento": detalles.get('birthday'),
            "fecha_fallecimiento": detalles.get('deathday'),
            "lugar_nacimiento": detalles.get('place_of_birth'),
            "departamento": detalles.get('known_for_department'),
            "genero": gender
        }

    else:
        #  3 Fallback a Wikipedia si no se encuentra en TMDB
        wiki = buscar_en_wikipedia(nombre)
        if isinstance(wiki, dict):
            return {
                "nombre": nombre,
                "biografia": wiki.get("summary", "Sin biografía disponible"),
                "url_wikipedia": wiki.get("url")
            }
        else:
            return f"No se encontró información sobre '{nombre}'."

    


<h2>Agente Cine</h2>

In [11]:
# Crear agente con ADK
agente_cine = Agent(
    name="AgenteCine",
    model=MODEL_GEMINI_2_0_FLASH,
    description="Agente experto en películas, consulta TMDB y ChromaDB",
    instruction=(
        "Eres un asistente experto en películas. "
        "Puedes buscar películas por título en TMDB y almacenarlas en ChromaDB, "
        "o consultar películas almacenadas según una descripción o título. "
        "Cuando el usuario pida buscar películas, usa 'buscar_y_guardar_peliculas'. "
        "Cuando el usuario pida consultar películas, usa 'obtener_peliculas_por_titulo' o 'obtener_detalles_pelicula'."
        "Si el usuario pregunta algo sobre el argumento de la pelicula o sus reseñas (reviews) , usa 'obtener_sinopsis_pelicula'"
        "Si el resultado obtenido contiene informacion de wikipedia , filtra y devuelve al usuario solo la informacion solicitada , por ejemplo si pregunta por un director , devuelve el director de esa pelicula ."
    ),
    tools=[buscar_y_guardar_peliculas, obtener_peliculas_por_titulo, obtener_detalles_pelicula,obtener_sinopsis_pelicula]
)

print(f"Agente '{agente_cine.name}' creado con modelo '{MODEL_GEMINI_2_0_FLASH}'.")


Agente 'AgenteCine' creado con modelo 'gemini-2.0-flash'.


<h2>Agente Actores</h2>

In [12]:
# Crear agente experto en actores
agente_actores = Agent(
    name="AgenteActores",
    model=MODEL_GEMINI_2_0_FLASH,
    description="Agente experto en actores y actrices, consulta TMDB, ChromaDB y Wikipedia",
    instruction=(
        "Eres un asistente experto en actores y actrices. "
        "Puedes buscar actores por nombre en TMDB y almacenarlos en ChromaDB, "
        "o consultar actores almacenados según un nombre o descripción. "
        "Cuando el usuario pida información sobre un actor, usa 'obtener_detalles_actor' y devuelvele toda la informacion pedida "
        "Si no pide ningun dato en concreto , devuelvele su nombre , edad , altura , año de nacimiento y participacion en al menos 3 peliculas"
        "Si el resultado obtenido contiene información de Wikipedia, filtra y devuelve solo lo solicitado, "
        "por ejemplo la fecha de nacimiento o biografía resumida."
    ),
    tools=[add_person_to_collection, obtener_peliculas_por_titulo, buscar_en_wikipedia]
)

print(f"Agente '{agente_actores.name}' creado con modelo '{MODEL_GEMINI_2_0_FLASH}'.")


Agente 'AgenteActores' creado con modelo 'gemini-2.0-flash'.


In [13]:
# Crear Runner y sesión
session_service = InMemorySessionService()
APP_NAME = "cine_app"
USER_ID = "Usuario"
SESSION_ID = "user"

import asyncio

# Crear sesión async
session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID
)
print(f"Session creada: App='{APP_NAME}', User='{USER_ID}', Session='{SESSION_ID}'")

#agente de cine
runner_cine = Runner(
    agent=agente_cine,
    app_name=APP_NAME,
    session_service=session_service
)
print(f"Runner creado para el agente '{runner_cine.agent.name}'.")

#agente de actores
runner_actores = Runner(
    agent=agente_actores,
    app_name=APP_NAME,
    session_service=session_service
)

print(f"Runner creado para el agente '{runner_actores.agent.name}'.")



Session creada: App='cine_app', User='Usuario', Session='user'
Runner creado para el agente 'AgenteCine'.
Runner creado para el agente 'AgenteActores'.


<h3>Funcion para llamar a cualquier agente</h3>

In [ ]:
# Función genérica para llamar a cualquier agente
async def call_any_agent_async(query: str, runner: Runner):
    print(f"\n>>> User Query: {query}")
    content = types.Content(role='user', parts=[types.Part(text=query)])
    final_response_text = "El agente no produjo respuesta final."

    async for event in runner.run_async(user_id=USER_ID, session_id=SESSION_ID, new_message=content):
        if event.is_final_response():
            if event.content and event.content.parts:
                final_response_text = event.content.parts[0].text
            break

    print(f"<<< Agent Response: {final_response_text}")


In [17]:
#hablar con el agente
await call_any_agent_async("Busca reviews de  Ready Player One",runner=runner_cine)
await call_any_agent_async("Dime informacion sobre LeonardoDiCaprio",runner=runner_actores)
await call_any_agent_async("Dime el reparto , director y fecha de estreno  de la pelicula Ready Player One",runner=runner_cine)




>>> User Query: Busca reviews de  Ready Player One


Event from an unknown agent: AgenteCine, event id: 45cc347c-3f31-461a-91c3-2a7757dd527c
Event from an unknown agent: AgenteCine, event id: af3598b1-93ef-4eee-9b86-56e5b33eafcb
Event from an unknown agent: AgenteCine, event id: bca04be0-558a-401c-be33-9ea376eccd61
Event from an unknown agent: AgenteCine, event id: 9c491875-fd97-405c-9d0b-b4840064eb85
Event from an unknown agent: AgenteCine, event id: 66980920-d3d1-4a80-b19d-30a541a85a12
Event from an unknown agent: AgenteCine, event id: e7f1ea19-d968-4ff9-8b6c-e9cbe9035fdd


<<< Agent Response: Ready Player One es una película de ciencia ficción estadounidense de 2018 dirigida por Steven Spielberg a partir de un guion de Ernest Cline y Zak Penn. Basada en la novela de Cline de 2011 Ready Player One, está protagonizada por Tye Sheridan, Olivia Cooke, Ben Mendelsohn, Lena Waithe, T.J. Miller, Simon Pegg y Mark Rylance. La película está ambientada en 2045, donde gran parte de la humanidad utiliza OASIS, una simulación de realidad virtual, para escapar del mundo real.


>>> User Query: Dime informacion sobre LeonardoDiCaprio


Event from an unknown agent: AgenteActores, event id: 3ad90193-c76f-4816-80c9-68e50201eee4


<<< Agent Response: Leonardo DiCaprio nació el 11 de noviembre de 1974. Es un actor y productor de cine estadounidense, conocido por sus papeles en películas biográficas y de época. Ha ganado numerosos premios, incluyendo un Oscar, un BAFTA y tres Globos de Oro. Algunas de sus películas más famosas son "Titanic", "El lobo de Wall Street" y "El renacido".


>>> User Query: Dime el reparto , director y fecha de estreno  de la pelicula Ready Player One
<<< Agent Response: El reparto de Ready Player One incluye a Tye Sheridan, Olivia Cooke, Ben Mendelsohn, Lena Waithe, T.J. Miller, Simon Pegg y Mark Rylance. La película fue dirigida por Steven Spielberg y se estrenó en 2018.

